# 노트북 역할 설명
## 해당 노트북은 법인별 수작업 데이터를 하나의 공통된 표준 데이터마트로 합산하기 위한 파일입니다.
### 법인별로 각자 다른 포맷의 엑셀파일을 활용하고 있으므로, 법인별 수작업 데이터의 형태를 파악하기 위해서는 'manual_data_by_entity.md' 파일을 참고하세요.


### ✅ '의뢰' 도메인에 대한 수작업 데이터 표준 데이터마트 생성
#### '의뢰' 표준 데이터 마트 컬럼 구성
- req_type_nm : 의뢰 접수 타입
- dev_comp_id : 개발법인 코드
- req_nation_cd : 의뢰법인 코드
- request_date : 의뢰 날짜
- gcc_category : GCC(Global Control Center)에서 관리하는 카테고리 대분류 기준
- proc_status : 의뢰 상태
- product_code : 신제품접수코드
- bulk_cod : 벌크구분코드
- mitem : 벌크코드
- item2: 완제품코드
- forml_code: 소제형코드
- product_name: 접수 제품명(신제품접수코드 이름)
- bulk_name: 접수 벌크명 (벌크구분코드 이름)
- customer_name : 고객사명
- brand_name : 브랜드명
- last_labno: 최종 랩넘버
- requester_id: 의뢰법인 담당연구원 사번
- requester_name: 의뢰법인 담당연구원명
- req_teamnm: 의뢰법인 담당연구원팀명
- developer_id: 개발법인 담당연구원 사번
- developer_name: 개발법인 담당연구원명
- dev_teamnm: 개발법인 담당연구원팀명
- reg_id: 의뢰법인의 등록자 id
- reg_nm: 의뢰법인의 등록자 명
- reg_deptid: 의뢰법인의 등록자 팀id
- reg_team: 의뢰법인 등록자 팀
- req_id: 의뢰법인의 마케팅 담당자 id
- req_nm: 의뢰법인의 마케팅 담당자 명
- req_team: 의뢰법인의 마케팅 팀
- req_num: 의뢰번호
- del_yn: 삭제여부
- memo: 비고

#### 표준 데이터 마트 생성 순서
1. 모든 법인에 대한 수작업 '의뢰'데이터를 로드한다. (각 법인 엑셀 데이터 중 의뢰 데이터 원본 시트만 불러오기)
2. 표준 데이터마트 컬럼(시스템DB 컬럼과 동일한 컬럼명을 사용) 형태로 모든 법인의 수작업 의뢰 데이터를 구성한다. (참고 'column_mapping_NPD.md' )


#### 표준 데이터 마트 생성 규칙
1. 표준 데이터마트 컬럼을 기준으로 수작업 '의뢰' 데이터를 재구성하되, 없는 정보는 빈 값으로 비워두면 됨
2. 신제품접수코드와 벌크구분코드는 추후 '시스템 DB'와 매핑할 가장 중요한 컬럼임. 띄어쓰기, 쉼표, 따옴표, 대쉬 등을 삭제하는 기본적인 전처리가 필요함.

---
## Step 1. 법인별 수작업 데이터 로드
> 각 법인의 엑셀 파일에서 의뢰(NPD) Raw Data 시트만 불러옵니다.
> 법인별 파일명 및 시트명은 `manual_data_by_entity.md` 참고

| 법인 | comp_id | 파일명 | Raw Data 시트명 |
|---|---|---|---|
| 상해 | 5100 | 상해_NPD.xlsx | `Raw Data` |
| 광저우 | 5200 | 광저우_수작업.xlsx | `전체` |
| 미국 | 6400 | 미국_수작업.xlsx | `US NPD request` |
| 태국 | 7200 | 태국_수작업.xlsx | `NPD 로우 데이터` |
| 인도네시아 | 7100 | 인니_수작업.xlsx | `Raw NPD Request 2026` |

In [46]:
import pandas as pd
import os

# 수작업 엑셀 파일이 저장된 기본 경로
DATA_DIR = "./manual_data"

In [47]:
# ─────────────────────────────────────────────
# 법인별 파일명 / NPD Raw Data 시트명 / 컬럼 헤더 행 정의
# header : 엑셀에서 실제 컬럼명이 위치한 행 (0-indexed)
# ─────────────────────────────────────────────
ENTITY_CONFIG = {
    "상해":   {"comp_id": "5100", "file": "상해_NPD.xlsx",      "sheet": "Raw Data",             "header": 1},
    "광저우": {"comp_id": "5200", "file": "광저우_수작업.xlsx",  "sheet": "전체",                  "header": 0},
    "미국":   {"comp_id": "6400", "file": "미국_수작업.xlsx",    "sheet": "US NPD request",       "header": 8},
    "태국":   {"comp_id": "7200", "file": "태국_수작업.xlsx",    "sheet": "NPD 로우 데이터",       "header": 2},
    "인니":   {"comp_id": "7100", "file": "인니_수작업.xlsx",    "sheet": "Raw NPD Request 2026", "header": 0},
}

# ─────────────────────────────────────────────
# 법인별 엑셀 로드 (의뢰 Raw Data 시트만 읽기)
# ─────────────────────────────────────────────
import warnings

raw_data = {}  # { 법인명: DataFrame }

for entity, cfg in ENTITY_CONFIG.items():
    file_path = os.path.join(DATA_DIR, cfg["file"])

    # openpyxl의 Data Validation 미지원 경고는 데이터에 영향 없으므로 무시
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        df = pd.read_excel(file_path, sheet_name=cfg["sheet"], header=cfg["header"], dtype=str)

    # 로드 결과 간단 확인 (행 수, 열 수, 첫 번째 컬럼명)
    print(f"[{entity}] comp_id={cfg['comp_id']} | shape={df.shape} | 첫 컬럼='{df.columns[0]}'")

    raw_data[entity] = df

[상해] comp_id=5100 | shape=(23210, 38) | 첫 컬럼='Unnamed: 0'
[광저우] comp_id=5200 | shape=(52188, 47) | 첫 컬럼='순번'
[미국] comp_id=6400 | shape=(3292, 21) | 첫 컬럼='Unnamed: 0'
[태국] comp_id=7200 | shape=(5888, 14) | 첫 컬럼='연도'
[인니] comp_id=7100 | shape=(2302, 13) | 첫 컬럼='Request date'


In [57]:
# 상해 실제 컬럼명 확인 (불일치 의심 컬럼 위주로 출력)
print("[미국] 전체 컬럼 목록:")
for col in raw_data["미국"].columns:
    print(f"  {repr(col)}")

[미국] 전체 컬럼 목록:
  'Unnamed: 0'
  'Unnamed: 1'
  'Unnamed: 2'
  'Year'
  'Request\nMonth'
  'Request\nDate'
  'Entity'
  'Sales name'
  'NPD\nProduct No'
  'NPD\nBulk No'
  'Customer'
  'Brand'
  'Product Name'
  'Bulk Name'
  'small class'
  'Small class code'
  'Team'
  'Chemist '
  'Status'
  'Category(2308)'
  'OTC\n(Sunscreen)'


In [55]:

df_gz = raw_data["광저우"].copy()

# 마지막 20행의 원본 컬럼 중 핵심 컬럼만 확인
print("=== 광저우 원본 마지막 20행 (핵심 컬럼) ===")
print(df_gz[["순번", "신제품접수코드", "벌크구분코드", "상태"]].tail(20).to_string())

# 전체에서 모든 컬럼이 NaN인 행 수 확인
all_nan_rows = df_gz.isna().all(axis=1).sum()
print(f"\n전체 행 중 모든 컬럼이 NaN인 행 수: {all_nan_rows}개")

# 첫 번째로 모든 컬럼이 NaN인 행의 인덱스
first_all_nan = df_gz[df_gz.isna().all(axis=1)].index[0]
print(f"첫 번째 전체 NaN 행 인덱스: {first_all_nan}")

=== 광저우 원본 마지막 20행 (핵심 컬럼) ===
          순번      신제품접수코드         벌크구분코드       상태
52168  52169  P9S12300010  P3S1230001001  샘플(1)진행
52169  52170  P9S12300009  P3S1230000902     처방확정
52170  52171  P9S12300009  P3S1230000901     처방확정
52171  52172  P9G12300001  P3G1230000101     배정대기
52172  52173  P9S12300008  P3S1230000801  샘플(1)제출
52173  52174  P9S12300007  P3S1230000701  샘플(1)제출
52174  52175  P9S12300006  P3S1230000601  샘플(1)제출
52175  52176  P9S12300005  P3S1230000501  샘플(1)제출
52176  52177  P9S12300004  P3S1230000401     처방확정
52177  52178  P9S12300003  P3S1230000301  샘플(1)진행
52178  52179  P9S12300002  P3S1230000201  샘플(1)제출
52179  52180  P9S12300001  P3S1230000101     처방등록
52180    NaN          NaN            NaN      NaN
52181    NaN          NaN            NaN      NaN
52182    NaN          NaN            NaN      NaN
52183    NaN          NaN            NaN      NaN
52184    NaN          NaN            NaN      NaN
52185    NaN          NaN            NaN      NaN
52186    NaN       

In [48]:
# 표준 데이터마트 컬럼 순서 정의 (시스템DB 컬럼명과 동일)
STANDARD_COLUMNS = [
    "req_type_nm", "dev_comp_id", "req_nation_cd", "request_date",
    "gcc_category", "proc_status", "product_code", "bulk_cod",
    "mitem", "item2", "forml_code", "product_name", "bulk_name",
    "customer_name", "brand_name", "last_labno",
    "requester_id", "requester_name", "req_teamnm",
    "developer_id", "developer_name", "dev_teamnm",
    "reg_id", "reg_nm", "reg_deptid", "reg_team",
    "req_id", "req_nm", "req_team", "req_num", "del_yn", "memo",
]


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    엑셀 컬럼명에 포함된 줄바꿈(\n)과 앞뒤 공백을 제거해 정규화한다.
    매핑 딕셔너리 key와 실제 파일 컬럼명의 불일치를 방지하기 위해 사용.
    """
    df.columns = df.columns.str.replace("\n", "", regex=False).str.strip()
    return df


def apply_concat_columns(df: pd.DataFrame, entity: str) -> pd.DataFrame:
    """
    CONCAT_COLUMNS 정의에 따라 복수 원본 컬럼을 | 구분자로 합산해 표준 컬럼에 저장한다.
    값이 NaN인 경우는 빈 문자열로 처리 후 합산하고, 전체가 빈 경우 NaN을 유지한다.
    """
    concat_map = CONCAT_COLUMNS.get(entity, {})
    for target_col, source_cols in concat_map.items():
        existing = [c for c in source_cols if c in df.columns]
        missing  = [c for c in source_cols if c not in df.columns]
        if missing:
            print(f"  ⚠ [{entity}] 합산 대상 컬럼 없음: {missing}")

        if existing:
            # 각 컬럼을 문자열로 변환 후 합산 (NaN → 빈 문자열)
            df[target_col] = (
                df[existing]
                .fillna("")
                .apply(lambda row: " | ".join(v for v in row if v != ""), axis=1)
                .replace("", pd.NA)   # 모든 값이 비어있으면 NaN 유지
            )

    return df


def to_standard_datamart(df: pd.DataFrame, entity: str) -> pd.DataFrame:
    """
    법인별 수작업 DataFrame을 표준 데이터마트 형태로 변환한다.

    처리 순서:
      1. 컬럼명 정규화 (\n 제거, 앞뒤 공백 제거)
      2. 복수 컬럼 합산 (CONCAT_COLUMNS 정의 기준, | 구분자)
      3. 매핑 딕셔너리 기준으로 컬럼명 rename
      4. 매핑되지 않은 표준 컬럼은 NaN으로 추가
      5. 표준 컬럼 순서로 정렬 후 반환
      * 변환 전 매핑 키 검증 (실제 파일에 없는 컬럼명 경고 출력)
    """
    mapping = COLUMN_MAPPING[entity]

    # ── 1. 컬럼명 정규화 ──────────────────────────────────────────────────
    df = normalize_columns(df.copy())

    # ── 2. 복수 컬럼 합산 ─────────────────────────────────────────────────
    df = apply_concat_columns(df, entity)

    # ── 검증: 매핑 key가 실제 DataFrame 컬럼에 존재하는지 확인 ──────────
    actual_cols = set(df.columns)
    missing_keys = [k for k in mapping if k not in actual_cols]
    if missing_keys:
        print(f"  ⚠ [{entity}] 매핑 key 불일치 (파일에 없는 컬럼): {missing_keys}")
    else:
        print(f"  ✓ [{entity}] 매핑 key 전체 일치")

    # ── 3. 매핑 기준으로 컬럼명 rename (실제 존재하는 key만 적용) ─────────
    valid_mapping = {k: v for k, v in mapping.items() if k in actual_cols}
    df_renamed = df.rename(columns=valid_mapping)

    # ── 4. 표준 컬럼 중 없는 컬럼은 NaN으로 추가 ─────────────────────────
    for col in STANDARD_COLUMNS:
        if col not in df_renamed.columns:
            df_renamed[col] = pd.NA

    # ── 5. 표준 컬럼 순서로 정렬 (비표준 원본 컬럼은 제외) ───────────────
    df_standard = df_renamed[STANDARD_COLUMNS].copy()

    return df_standard


print("변환 함수 정의 완료 : to_standard_datamart(df, entity)")

변환 함수 정의 완료 : to_standard_datamart(df, entity)


### 2-2. 표준 데이터마트 변환 함수 작성
> 법인별 DataFrame을 표준 컬럼 구조로 변환하는 함수
> 혼합 사용 컬럼 처리(2-3)는 이 함수 호출 전에 별도 수행

In [50]:
# ─────────────────────────────────────────────────────────────────────────────
# 법인별 수작업 컬럼명 → 표준 데이터마트 컬럼명 매핑 딕셔너리
# 기준 문서 : column_mapping_NPD.md
#
# 작성 규칙
#   - key   : 수작업 엑셀의 원본 컬럼명 (정규화 후 기준 — \n 제거, 앞뒤 공백 제거)
#   - value : 표준 데이터마트 컬럼명 (시스템DB 컬럼명과 동일)
#   - 매핑 없는 컬럼(-) 은 딕셔너리에서 제외 → 변환 후 NaN으로 자동 처리
#   - 복수 컬럼을 합산해야 하는 경우는 CONCAT_COLUMNS 에 별도 정의
#   - 혼합 사용 컬럼(1개 수작업 컬럼 → 2개 표준 컬럼)은 ※ 주석으로 표시,
#     실제 분리 로직은 Step 2-3 예외 처리에서 수행
# ─────────────────────────────────────────────────────────────────────────────

COLUMN_MAPPING = {

    # ── 상해 (comp_id: 5100) ──────────────────────────────────────────────
    "상해": {
        "구분":                   "req_type_nm",
        "의뢰일자":               "request_date",
        "카테고리":               "gcc_category",
        "상태":                   "proc_status",
        "신제품접수코드":         "product_code",
        "벌크구분코드":           "bulk_cod",
        "SAP 벌크코드":           "mitem",
        "SAP 제품코드":           "item2",
        "제형코드":               "forml_code",
        # ※ "제품명" → product_name 과 bulk_name 에 혼용 (2-3에서 처리)
        "고객사명":               "customer_name",
        "브랜드명":               "brand_name",
        "(의뢰법인)제형 연구원":  "requester_name",   # 정규화 후: \n 제거
        "(의뢰법인)제형팀":       "req_teamnm",        # 정규화 후: \n 제거
        "(개발법인)제형연구원":   "developer_name",    # 정규화 후: \n 제거
        "(개발법인)제형팀":       "dev_teamnm",        # 정규화 후: \n 제거
        "마케팅담당자":           "req_nm",
        "마케팅팀":               "req_team",
        "삭제여부":               "del_yn",
        "비고(是否储备品)":       "memo",              # 실제 컬럼명 반영
    },

    # ── 광저우 (comp_id: 5200) ────────────────────────────────────────────
    "광저우": {
        "구분":              "req_type_nm",
        "의뢰일자":          "request_date",
        "扩大":              "gcc_category",
        "상태":              "proc_status",
        "신제품접수코드":    "product_code",
        "벌크구분코드":      "bulk_cod",
        "SAP 벌크코드":      "mitem",
        "SAP 제품코드":      "item2",
        "제형코드":          "forml_code",
        "제품명":            "product_name",
        "벌크명":            "bulk_name",
        "고객사명":          "customer_name",
        "브랜드명":          "brand_name",
        "랩넘버":            "last_labno",
        "(의뢰법인)제형연구원": "requester_name",
        "(의뢰법인)제형팀":  "req_teamnm",
        "(개발법인)제형연구원": "developer_name",
        "(개발법인)제형팀":  "dev_teamnm",
        "마케팅담당자":      "req_nm",
        "마케팅팀":          "req_team",
        "삭제여부":          "del_yn",
        "Sample/Benchmark/Lab.#": "memo",
    },

    # ── 미국 (comp_id: 6400) ──────────────────────────────────────────────
    # memo → CONCAT_COLUMNS 에서 Entity | Status | OTC(Sunscreen) 합산 처리
    "미국": {
        "Request Date":      "request_date",
        "Category(2308)":    "gcc_category",
        "NPD Product No":    "product_code",
        "NPD Bulk No":       "bulk_cod",
        "Small class code":  "forml_code",
        "Product Name":      "product_name",
        "Bulk Name":         "bulk_name",
        "Customer":          "customer_name",
        "Brand":             "brand_name",
        "Chemist":           "developer_name",
        "Team":              "dev_teamnm",
        "Sales name":        "req_nm",
    },

    # ── 태국 (comp_id: 7200) ──────────────────────────────────────────────
    "태국": {
        "NPS Date":          "request_date",
        "회의용 분류":        "gcc_category",
        # ※ "NPD Code" → product_code(P9%) 와 bulk_cod(P3%) 혼용 (2-3에서 Prefix로 분리)
        # ※ "Product Name" → product_name 과 bulk_name 에 혼용 (2-3에서 처리)
        "Customer Name":     "customer_name",
        "Brand":             "brand_name",
        "Person in charge (MK)": "req_nm",
        "Sub Category":      "memo",
    },

    # ── 인도네시아 (comp_id: 7100) ────────────────────────────────────────
    # memo → CONCAT_COLUMNS 에서 Formulation | New/existing 합산 처리
    "인니": {
        "Classification":    "req_type_nm",
        "Request date":      "request_date",
        "Extended Category": "gcc_category",
        "NPD Product No":    "product_code",
        "NPD Bulk No":       "bulk_cod",
        "Bulk name":         "bulk_name",
        "Customer":          "customer_name",
        "Brand":             "brand_name",
        "Sales In Charge":   "req_nm",
        "Sales team":        "req_team",
    },
}

# ─────────────────────────────────────────────────────────────────────────────
# 복수 컬럼을 하나의 표준 컬럼에 | 구분자로 합산하는 매핑 정의
# 기준 문서 : column_mapping_NPD.md > 비고 컬럼 섹션
#
# 구조 : { 법인명: { 표준컬럼명: [합산할 원본 컬럼명 리스트] } }
# ─────────────────────────────────────────────────────────────────────────────
CONCAT_COLUMNS = {
    "미국": {
        "memo": ["Entity", "Status", "OTC(Sunscreen)"],
    },
    "인니": {
        "memo": ["Formulation", "New/existing"],
    },
}

print("컬럼 매핑 딕셔너리 정의 완료")
for entity, mapping in COLUMN_MAPPING.items():
    concat_info = CONCAT_COLUMNS.get(entity, {})
    print(f"  [{entity}] 단일 매핑: {len(mapping)}개 | 합산 매핑: {len(concat_info)}개")

컬럼 매핑 딕셔너리 정의 완료
  [상해] 단일 매핑: 19개 | 합산 매핑: 0개
  [광저우] 단일 매핑: 22개 | 합산 매핑: 0개
  [미국] 단일 매핑: 12개 | 합산 매핑: 1개
  [태국] 단일 매핑: 6개 | 합산 매핑: 0개
  [인니] 단일 매핑: 10개 | 합산 매핑: 1개


In [56]:
to_standard_datamart(raw_data['미국'],'미국')

  ⚠ [미국] 매핑 key 불일치 (파일에 없는 컬럼): ['Request Date', 'NPD Product No', 'NPD Bulk No']


,req_type_nm,dev_comp_id,req_nation_cd,request_date,gcc_category,proc_status,product_code,bulk_cod,mitem,item2,...,reg_id,reg_nm,reg_deptid,reg_team,req_id,req_nm,req_team,req_num,del_yn,memo
0,<NA>,<NA>,<NA>,<NA>,크림류,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,In progress
1,<NA>,<NA>,<NA>,<NA>,스킨류,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,In progress
2,<NA>,<NA>,<NA>,<NA>,크림류,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,In progress
3,<NA>,<NA>,<NA>,<NA>,클렌징,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,In progress
4,<NA>,<NA>,<NA>,<NA>,립 메이크업,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,In progress
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3287,<NA>,<NA>,<NA>,<NA>,스킨류,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,Tiffany Ryu,<NA>,<NA>,<NA>,CUSA
3288,<NA>,<NA>,<NA>,<NA>,스킨류,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,Tiffany Ryu,<NA>,<NA>,<NA>,CUSA
3289,<NA>,<NA>,<NA>,<NA>,스킨류,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,Heejung Kim,<NA>,<NA>,<NA>,CCC
3290,<NA>,<NA>,<NA>,<NA>,스킨류,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,Sahar Nessary,<NA>,<NA>,<NA>,CCC


---
## Step 2. 표준 데이터마트 생성

### 2-1. 법인별 컬럼 매핑 딕셔너리 정의
> `column_mapping_NPD.md` 기준으로 수작업 컬럼명 → 표준 컬럼명 매핑
> `-` (매핑 없음) 컬럼은 딕셔너리에서 제외 → 변환 후 NaN으로 처리
> 혼합 사용 컬럼(태국 `NPD Code`, 상해/태국 `제품명`/`Product Name`)은 2-3에서 별도 처리